# Train Card Classifier on Google Colab

This notebook trains a YOLOv8 classification model on your sorted card dataset.

**Instructions:**
1. Upload this notebook to Google Colab
2. Enable GPU: Runtime → Change runtime type → GPU (T4)
3. Upload your `sorted.zip` file (or create it first)
4. Run all cells

## Step 1: Setup Environment

In [ ]:
# Install YOLOv8
!pip install ultralytics -q

# Verify GPU is available
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: No GPU detected! Training will be slow.")
    print("Go to Runtime → Change runtime type → GPU")

## Step 2: Upload Your Sorted Dataset

**Option A: Upload sorted.zip from your computer**

Before running this notebook, create the zip file on your Mac:
```bash
cd /Users/eden/Desktop/clash-royale-rl
zip -r sorted.zip sorted/
```

Then upload it using the file browser on the left, or run the cell below to upload.

In [ ]:
from google.colab import files
import os

# Upload sorted.zip
print("Please upload your sorted.zip file:")
uploaded = files.upload()

# Unzip dataset
!unzip -q sorted.zip
print("✅ Dataset uploaded and extracted")

# Verify dataset structure
!ls sorted/ | head -10

## Step 3: Check Dataset

Verify your dataset looks correct before training.

In [ ]:
import os
from pathlib import Path

# Count images per class
sorted_dir = Path('sorted')
class_counts = {}

for class_dir in sorted(sorted_dir.iterdir()):
    if class_dir.is_dir():
        image_count = len(list(class_dir.glob('*.png'))) + len(list(class_dir.glob('*.jpg')))
        class_counts[class_dir.name] = image_count

# Print summary
print("Dataset Summary:")
print("=" * 50)
total_images = 0
for class_name, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{class_name:20} {count:4} images")
    total_images += count

print("=" * 50)
print(f"Total Classes: {len(class_counts)}")
print(f"Total Images: {total_images}")
print("\n✅ Dataset looks good!" if total_images > 0 else "❌ No images found!")

## Step 4: Train Card Classifier

Train YOLOv8 classification model on your dataset.

**Training parameters:**
- `epochs=50` - Maximum epochs (will stop early if no improvement)
- `imgsz=64` - Image size (cards are small)
- `batch=32` - Batch size (adjust if GPU runs out of memory)
- `patience=10` - Early stopping patience

**Estimated time:** ~10-20 minutes on Colab GPU

In [ ]:
from ultralytics import YOLO

# Initialize model
model = YOLO('yolov8n-cls.pt')

# Train
results = model.train(
    data='sorted',
    epochs=50,
    imgsz=64,
    batch=32,
    patience=10,
    device=0,  # Use GPU
    name='card_classifier',
    project='runs/classify'
)

print("\n" + "="*80)
print("✅ TRAINING COMPLETE!")
print("="*80)

## Step 5: View Training Results

In [ ]:
from IPython.display import Image, display

# Display training curves
print("Training Results:")
print("=" * 80)

# Show confusion matrix
try:
    display(Image('runs/classify/card_classifier/confusion_matrix.png'))
except:
    print("Confusion matrix not available")

# Show training curves
try:
    display(Image('runs/classify/card_classifier/results.png'))
except:
    print("Results plot not available")

## Step 6: Test the Model

Test predictions on some sample images to verify the model works.

In [ ]:
# Load best model
best_model = YOLO('runs/classify/card_classifier/weights/best.pt')

# Test on a few images
test_class = list(class_counts.keys())[0]  # Get first class
test_images = list(Path(f'sorted/{test_class}').glob('*.png'))[:3]

print(f"Testing on {test_class} images:")
print("=" * 80)

for img_path in test_images:
    results = best_model.predict(img_path, verbose=False)
    
    # Get top prediction
    probs = results[0].probs
    top1_class = results[0].names[probs.top1]
    top1_conf = probs.top1conf.item()
    
    correct = "✅" if top1_class == test_class else "❌"
    print(f"{correct} {img_path.name}: {top1_class} ({top1_conf:.2%})")

print("\n✅ Model testing complete!")

## Step 7: Download Trained Model

Download the trained model to use on your Mac.

In [ ]:
from google.colab import files

# Download best model
print("Downloading best model...")
files.download('runs/classify/card_classifier/weights/best.pt')

print("\n" + "="*80)
print("✅ MODEL DOWNLOADED!")
print("="*80)
print("\nNext steps on your Mac:")
print("1. Move the downloaded 'best.pt' to: models/card_classifier.pt")
print("   cd /Users/eden/Desktop/clash-royale-rl")
print("   mv ~/Downloads/best.pt models/card_classifier.pt")
print("\n2. Test the new classifier:")
print("   python3 main.py --model --games 1")
print("\n3. Start RL training:")
print("   python3 main.py --rl --games 100")

## Optional: Download Full Training Results

In [ ]:
# Zip all training results
!zip -r training_results.zip runs/classify/card_classifier/

# Download
files.download('training_results.zip')

print("✅ Full training results downloaded!")